# PROJECT ZARA: REPRODUCIBLE VOICE MODEL TRAINING & FINE-TUNING PIPELINE

**Platform**: Google Colab / Cloud GPU (NVIDIA T4, A10G, or A100)  
**Objective**: Fine-tune an expressive, code-switched conversational speech model with consistent persona timbre, dynamic tone steering (`[happy]`, `[whisper]`), and non-verbal paralinguistics (`<laugh>`, `<sigh>`).  
**Target Architecture**: Causal SpeechLM / LoRA Parameter-Efficient Adaptation with Kyutai Mimi 24kHz RVQ Codec.

## 1. Hardware Inspection & GPU Diagnostics
We inspect the active GPU, CUDA compatibility, and available VRAM. A minimum of 12 GB VRAM is recommended for FP16 LoRA fine-tuning.

In [ ]:
import os
import sys
import torch

print("=== Environment Diagnostics ===")
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {gpu_name}")
    print(f"Allocated VRAM: {vram_gb:.2f} GB")
    if vram_gb < 11.5:
        print("⚠️ Notice: Less than 12 GB VRAM detected. Enable 8-bit QLoRA quantization.")
    else:
        print("✓ Hardware confirmed suitable for full FP16 LoRA fine-tuning.")
else:
    print("⚠️ No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU or A100.")

## 2. Google Drive Mounting & Persistent Checkpoint Storage
Ephemeral Colab environments disconnect when idle. We mount Google Drive to persist trained checkpoints, training state, and exported models permanently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSISTENT_DIR = "/content/drive/MyDrive/zara_voice_training"
CHECKPOINT_DIR = f"{PERSISTENT_DIR}/checkpoints"
DATASET_DIR = f"{PERSISTENT_DIR}/dataset"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"✓ Persistent storage initialized at: {PERSISTENT_DIR}")

## 3. Dependency Installation
Install the required speech processing and deep learning libraries.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes soundfile pydub edge-tts librosa

## 4. Dataset Preparation & Acoustic Normalization
We create a representative expressive speech dataset with paired emotional transcripts (`[happy]`, `[whisper]`, `<laugh>`, `<sigh>`). In production, point `RAW_AUDIO_DIR` to your studio voice recordings.

In [ ]:
# Synthesize clean initial training audio baseline using Zara Expressive Engine
import asyncio
import edge_tts
import csv

RAW_DIR = "./raw_recordings"
os.makedirs(RAW_DIR, exist_ok=True)

training_corpus = [
    ("zara_001", "[happy] Arre wah! I am so excited to talk with you today! <laugh>", "happy"),
    ("zara_002", "[whisper] It's really late at night, let's keep things soft and quiet.", "whisper"),
    ("zara_003", "[sad] That sounds really exhausting... <sigh> sometimes everything feels heavy.", "sad"),
    ("zara_004", "[happy] You worked so hard on this project, I am genuinely proud of you!", "happy"),
    ("zara_005", "Bilkul tension mat lo yaar, we will figure out a proper jugaad together.", "neutral"),
    ("zara_006", "[whisper] Just breathe slowly. Tomorrow is a brand new day.", "whisper"),
    ("zara_007", "[happy] That joke was hilarious! <laugh> Tell me another one!", "happy"),
    ("zara_008", "I'm always here to listen whenever you need to talk.", "neutral"),
    ("zara_009", "[sad] It's okay to feel down sometimes... <sigh> don't be so hard on yourself.", "sad"),
    ("zara_010", "Chai peete hain aur relax karte hain, bohot kaam ho gaya aaj.", "neutral")
]

async def generate_training_samples():
    metadata_rows = []
    for uid, text, tone in training_corpus:
        out_wav = f"{RAW_DIR}/{uid}.mp3"
        comm = edge_tts.Communicate(text, "en-IN-NeerjaNeural")
        await comm.save(out_wav)
        metadata_rows.append({"id": uid, "audio_file": f"{uid}.mp3", "text": text, "tone": tone})
    
    with open(f"{RAW_DIR}/metadata.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "audio_file", "text", "tone"])
        writer.writeheader()
        writer.writerows(metadata_rows)
    print(f"✓ Generated {len(metadata_rows)} training samples with metadata.csv")

await generate_training_samples()

### Run Dataset Preprocessing Pipeline
Downmix to mono, resample to 24 kHz, normalize to -23 LUFS, trim leading/trailing silences, and emit `train_manifest.jsonl` and `val_manifest.jsonl`.

In [ ]:
!python training/voice/prepare_dataset.py \
  --input_dir ./raw_recordings \
  --output_dir /content/processed_dataset \
  --target_sr 24000 \
  --normalize_lufs -23.0

## 5. Dataset Validation & Manifest Inspection
Inspect the processed train and validation manifests.

In [ ]:
import json

with open("/content/processed_dataset/train_manifest.jsonl", "r", encoding="utf-8") as f:
    train_samples = [json.loads(line) for line in f]

print(f"✓ Total Training Samples: {len(train_samples)}")
print("Sample Entry:")
print(json.dumps(train_samples[0], indent=2))

## 6. Configure & Launch LoRA Fine-Tuning
We configure LoRA rank $r=16$, learning rate $2\times 10^{-4}$ with cosine annealing, mixed precision, and gradient accumulation.

In [ ]:
config = {
    "experiment_name": "zara_colab_pilot_v1",
    "base_model": "rumik-ai/rumik-oss-1",
    "lora_r": 16,
    "lora_alpha": 32,
    "learning_rate": 2e-4,
    "num_train_epochs": 5,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "logging_steps": 2
}

with open("/content/pilot_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

# Run training engine
!python training/voice/train_voice.py \
  --config /content/pilot_config.json \
  --train_manifest /content/processed_dataset/train_manifest.jsonl \
  --val_manifest /content/processed_dataset/val_manifest.jsonl \
  --output_dir {CHECKPOINT_DIR}/zara_pilot_v1 \
  --epochs 5

## 7. Evaluate Checkpoint on Standardized Benchmarks
Run automated acoustic evaluations: duration, RTF, SNR, and tone tag adherence.

In [ ]:
!python training/voice/evaluate_voice.py \
  --output_dir /content/eval_results \
  --voice_name zara-expressive

## 8. In-Notebook Audio Playback & Perceptual Check
Listen directly to the generated benchmark audio samples.

In [ ]:
from IPython.display import Audio, display

eval_report_path = "/content/eval_results/eval_report.json"
if os.path.exists(eval_report_path):
    with open(eval_report_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    print(f"Overall Evaluation RTF: {eval_data['overall_rtf']} | Realtime: {eval_data['realtime_capable']}")
    print(f"Average Signal-to-Noise Ratio: {eval_data['average_snr_db']} dB\n")
    
    for bench in eval_data["benchmarks"]:
        print(f"=== {bench['category'].upper()} ===")
        print(f"Text: {bench['raw_text']}")
        print(f"Tone: {bench['detected_tone']} | RTF: {bench['rtf']} | SNR: {bench['snr_db']} dB")
        display(Audio(bench['wav_path']))
else:
    print("Evaluation report not found.")

## 9. Export & Package Model Artifacts for Local Deployment
Copy trained artifacts directly to Google Drive for permanent backup.

In [ ]:
!cp -r /content/eval_results {PERSISTENT_DIR}/evaluation_summary
print(f"✓ Complete model artifacts and evaluation benchmarks successfully saved to Google Drive!")
print(f"Artifact Path: {PERSISTENT_DIR}")